## Real-Time Fraud Detection Pipeline
### Auto Loader → Bronze → Silver → AI Scoring

### Scenario
Your fintech platform processes **millions of transaction events** daily — payments, transfers, refunds, and fraud checks. These events originate from an upstream system (Amazon Kinesis) and land as JSON files in cloud storage.

Your job: build a **full medallion pipeline** — from raw file ingestion through real-time enrichment to AI-powered fraud scoring — using Auto Loader, Structured Streaming, and Model Serving on Databricks.

### Three-Notebook Architecture
This demo uses **three notebooks** — just like a real decoupled system:

| Notebook | Role | When to Run |
|---|---|---|
| [Kinesis Event Producer Simulator](#notebook-2466812635894404) | Continuously writes JSON event files to the landing zone (simulates Kinesis Firehose) | **Tab 1** — keep running |
| **This notebook** (Consumer Pipeline) | Ingests, enriches, and scores transactions end-to-end | **Tab 2** — you're here |
| [Fraud Detection Model Setup](#notebook-2466812635894406) | Trains a fraud model, registers to Unity Catalog, deploys a serving endpoint | **One-time** — run before Step 7 |

### What You'll Learn

| Step | Topic | Key Concept |
|---|---|---|
| **0** | Environment Setup | Catalog, schema, and Volume creation |
| **1** | Peek at Landing Zone | Inspect raw JSON files from the producer |
| **2** | Auto Loader Bronze Ingestion | `cloudFiles` format, schema inference, `Trigger.AvailableNow` |
| **3** | Verify Bronze | Row counts, schema inspection, event distribution |
| **4** | Incremental Pickup | Checkpoint-based exactly-once file discovery |
| **5** | Schema Evolution | `addNewColumns` + `mergeSchema` for upstream changes |
| **6** | Silver Enrichment | Streaming transforms, deduplication, derived fraud features |
| | Velocity Checks | 1-minute windowed aggregations with watermarks |
| | Channel Fingerprinting | Multi-channel anomaly detection per sender |
| **7** | AI Fraud Scoring | `ai_query()` via serving endpoint + Spark UDF alternative |
| **8** | Cleanup | Tear down all demo resources |

### Getting Started
1. **Open the [Kinesis Event Producer Simulator](#notebook-2466812635894404)** in a new browser tab
2. Run cells 1–3 in that notebook to start the continuous event producer
3. Come back here and proceed from **Step 0** below
4. Before reaching **Step 7**, run the [Fraud Detection Model Setup](#notebook-2466812635894406) notebook to train and deploy the scoring endpoint (∼5 min)

### Step 0 — Environment Setup
We create a dedicated **catalog**, **schema**, and **managed Volume** to isolate this demo. The Volume at `/Volumes/fintech_demo/bronze_layer/kinesis_landing` acts as our simulated Kinesis Firehose delivery path.

> **Instructor note:** In production, this Volume path would be an external Volume pointing to the S3 prefix where Kinesis Data Firehose writes JSON.

> **Run this cell first**, then start the Producer notebook in another tab before proceeding.

In [0]:
# ---------------------------------------------------------------------------
# Shared configuration — MUST match the producer notebook
# ---------------------------------------------------------------------------
CATALOG   = "fintech_demo"
SCHEMA    = "bronze_layer"
VOLUME    = "kinesis_landing"          # simulated Kinesis Firehose delivery path
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.txn_events_bronze"

# Checkpoint stored inside the Volume (fine for demo; use a separate path in prod)
CHECKPOINT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/_checkpoint"

# Landing zone where JSON files arrive
LANDING_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/events"

print(f"\u2705 Catalog  : {CATALOG}")
print(f"\u2705 Schema   : {CATALOG}.{SCHEMA}")
print(f"\u2705 Volume   : /Volumes/{CATALOG}/{SCHEMA}/{VOLUME}")
print(f"\u2705 Landing  : {LANDING_PATH}")
print(f"\u2705 Bronze   : {BRONZE_TABLE}")

In [0]:
# Ensure the landing zone exists (producer notebook creates the Volume)
import os
os.makedirs(LANDING_PATH, exist_ok=True)

print(f"\u2705 Producer configured")
print(f"   Landing zone: {LANDING_PATH}")

### Step 1 — Peek at the Landing Zone
Before we ingest, let’s see what the Producer notebook has dropped into our landing zone. If the producer is running, you should see JSON files accumulating. Each file contains 50 newline-delimited JSON events — exactly what Kinesis Data Firehose would deliver.

> **Re-run this cell** at any time to see new files appear.

In [0]:
# ---------------------------------------------------------------------------
# Check what files the Producer notebook has delivered so far
# ---------------------------------------------------------------------------
import os

try:
    files = sorted([f for f in os.listdir(LANDING_PATH) if f.endswith(".json")])
    print(f"\U0001F4C2 Landing zone: {LANDING_PATH}")
    print(f"   Total JSON files: {len(files)}\n")
    for f in files[-10:]:  # show last 10 files
        size = os.path.getsize(os.path.join(LANDING_PATH, f))
        print(f"   \u2022 {f}  ({size:,} bytes)")
    if len(files) > 10:
        print(f"   ... and {len(files) - 10} earlier files")
    if len(files) == 0:
        print("   \u26a0\ufe0f  No files yet! Make sure the Producer notebook is running in another tab.")
except FileNotFoundError:
    print("\u26a0\ufe0f  Landing zone not found. Run Step 0 first, then start the Producer notebook.")

### Step 2 — Ingest into Bronze with Auto Loader

This is the core of the demo. We use `cloudFiles` (Auto Loader) to:

1. **Discover** all JSON files in the landing zone
2. **Infer the schema** automatically (no manual DDL needed)
3. **Add ingestion metadata** — `_metadata.file_path` and `_metadata.file_modification_time` come free with Auto Loader
4. **Write to a Delta table** with exactly-once guarantees via checkpointing

Key options:
* `cloudFiles.format` — `json` (matches Kinesis Firehose NDJSON output)
* `cloudFiles.schemaLocation` — persists the inferred schema so it's stable across runs
* `cloudFiles.inferColumnTypes` — infer actual types instead of defaulting everything to STRING
* `trigger(availableNow=True)` — processes all pending files then stops (required on serverless)

> **Re-run the cell below** each time you want to catch up with the producer. Each run picks up only NEW files.

In [0]:
# ---------------------------------------------------------------------------
# Auto Loader — Bronze ingestion from the Kinesis landing zone
# ---------------------------------------------------------------------------
# This is the CORE cell of the demo. It uses cloudFiles (Auto Loader) to:
#   1. Discover all new JSON files in the landing zone
#   2. Infer the schema automatically
#   3. Add audit columns (_ingest_timestamp, _source)
#   4. Write to a Bronze Delta table with exactly-once guarantees
#
# Re-run this cell anytime to pick up NEW files the producer has dropped.
# The checkpoint ensures already-processed files are never re-ingested.
# ---------------------------------------------------------------------------
from pyspark.sql.functions import current_timestamp, lit

raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", CHECKPOINT)
    .option("cloudFiles.inferColumnTypes", "true")       # infer real types
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  # handle new fields
    .load(LANDING_PATH)
)

# Add Bronze audit columns
bronze_stream = (
    raw_stream
    .withColumn("_ingest_timestamp", current_timestamp())
    .withColumn("_source", lit("kinesis_firehose_sim"))
)

# Write to Bronze Delta table
query = (
    bronze_stream.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT)
    .option("mergeSchema", "true")              # allow schema evolution on write
    .trigger(availableNow=True)                  # serverless-compatible trigger
    .toTable(BRONZE_TABLE)
)

# Wait for the micro-batch to finish
query.awaitTermination()
print(f"\u2705 Auto Loader stream completed")
print(f"   Bronze table: {BRONZE_TABLE}")

### Step 3 — Verify the Bronze Table
Let’s inspect what Auto Loader wrote. Notice the automatically inferred schema, the audit columns, and the event count matching what the producer delivered.

In [0]:
# ---------------------------------------------------------------------------
# Quick Bronze table inspection
# ---------------------------------------------------------------------------
bronze_df = spark.table(BRONZE_TABLE)

print(f"\U0001F4CA Row count: {bronze_df.count():,}")
print(f"\n\U0001F4DD Schema:")
bronze_df.printSchema()

In [0]:
# ---------------------------------------------------------------------------
# Preview a sample of Bronze records
# ---------------------------------------------------------------------------
display(
    spark.sql(f"""
        SELECT event_id, event_type, timestamp, amount, currency,
               sender_id, receiver_id, merchant, status, channel,
               _ingest_timestamp, _source
        FROM {BRONZE_TABLE}
        ORDER BY timestamp DESC
        LIMIT 1000
    """)
)

In [0]:
# ---------------------------------------------------------------------------
# Quick aggregation — event distribution (useful for fintech monitoring)
# ---------------------------------------------------------------------------
display(
    spark.sql(f"""
        SELECT event_type,
               status,
               COUNT(*)           AS event_count,
               ROUND(AVG(amount), 2) AS avg_amount,
               ROUND(SUM(amount), 2) AS total_volume
        FROM {BRONZE_TABLE}
        GROUP BY event_type, status
        ORDER BY event_type, status
    """)
)

### Step 4 — Incremental Pickup

This is where Auto Loader shines. The producer notebook is **still running** in your other tab, continuously dropping new files.

1. **Capture the current row count** (cell below)
2. **Wait 30–60 seconds** for the producer to drop more files
3. **Re-run the Auto Loader cell** (Step 2) to pick up only the *new* files
4. **Re-run the verify cell** to see the count increase

The checkpoint tracks which files have already been processed — no duplicates, no reprocessing.

In [0]:
# ---------------------------------------------------------------------------
# Capture the BEFORE count, then re-run Auto Loader to ingest new files
# ---------------------------------------------------------------------------
from pyspark.sql.functions import current_timestamp, lit

before_count = spark.table(BRONZE_TABLE).count()
print(f"\U0001F4CA Before count: {before_count:,} rows")
print(f"   Re-running Auto Loader to pick up new files from producer...\n")

raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", CHECKPOINT)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(LANDING_PATH)
)

bronze_stream = (
    raw_stream
    .withColumn("_ingest_timestamp", current_timestamp())
    .withColumn("_source", lit("kinesis_firehose_sim"))
)

query = (
    bronze_stream.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(BRONZE_TABLE)
)

query.awaitTermination()

after_count = spark.table(BRONZE_TABLE).count()
new_events = after_count - before_count

print(f"\u2705 Incremental stream completed")
print(f"   Before: {before_count:,} | After: {after_count:,} | New: {new_events:,}")
if new_events > 0:
    print(f"   \U0001F389 Auto Loader picked up only the NEW files!")
else:
    print(f"   \u2139\ufe0f  No new files since last run. Wait for the producer to drop more, then re-run.")

In [0]:
# ---------------------------------------------------------------------------
# Show ingestion timeline — you should see distinct micro-batches
# ---------------------------------------------------------------------------
print(f"\U0001F4CA Current Bronze row count: {spark.table(BRONZE_TABLE).count():,}\n")

# Ingestion timeline shows each Auto Loader run as a distinct batch
display(
    spark.sql(f"""
        SELECT DATE_TRUNC('second', _ingest_timestamp) AS ingest_second,
               COUNT(*) AS events_ingested
        FROM {BRONZE_TABLE}
        GROUP BY 1
        ORDER BY 1
    """)
)

### Step 5 — Schema Evolution

Upstream systems evolve. Now we’ll simulate the payments team adding a `risk_score` field to events.

**Instructions:**
1. Go to the **Producer notebook** tab
2. **Interrupt** the running cell (■ stop button)
3. **Run the Schema Evolution cell** (the last code cell in that notebook) — it produces events with a new `risk_score` field
4. Come back here and **run the cell below** to ingest the new events

Auto Loader’s **`addNewColumns`** mode detects the new field and merges it into the table schema automatically. Combined with `mergeSchema` on the write side, the Bronze table grows to accommodate the change — zero downtime, zero manual DDL.

> **Key insight:** In a medallion architecture, the Bronze table is schema-flexible by design. Downstream Silver/Gold layers enforce stricter contracts.

In [0]:
# ---------------------------------------------------------------------------
# Re-run Auto Loader — schema evolution kicks in automatically
# The producer is now writing events with a "risk_score" field.
# Auto Loader detects the new column and merges it into the Bronze table.
# ---------------------------------------------------------------------------
from pyspark.sql.functions import current_timestamp, lit

before_count = spark.table(BRONZE_TABLE).count()

raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", CHECKPOINT)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(LANDING_PATH)
)

bronze_stream = (
    raw_stream
    .withColumn("_ingest_timestamp", current_timestamp())
    .withColumn("_source", lit("kinesis_firehose_sim"))
)

query = (
    bronze_stream.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(BRONZE_TABLE)
)

query.awaitTermination()

after_count = spark.table(BRONZE_TABLE).count()
print(f"\u2705 Schema evolution stream completed")
print(f"   Ingested {after_count - before_count:,} new events (total: {after_count:,})")

In [0]:
# ---------------------------------------------------------------------------
# Verify schema evolution: risk_score should now appear in the table
# ---------------------------------------------------------------------------
final_count = spark.table(BRONZE_TABLE).count()

print(f"\U0001F4CA Total Bronze row count: {final_count:,}")
print(f"\n\U0001F4DD Updated schema (note the new 'risk_score' column):")
spark.table(BRONZE_TABLE).printSchema()

# Show that older rows have NULL for risk_score, newer rows have values
print("\n\U0001F50D risk_score distribution (NULL = pre-evolution, NOT NULL = post-evolution):")
display(
    spark.sql(f"""
        SELECT CASE WHEN risk_score IS NOT NULL THEN 'Has risk_score' ELSE 'No risk_score (pre-evolution)' END AS schema_version,
               COUNT(*) AS event_count,
               ROUND(AVG(amount), 2) AS avg_amount
        FROM {BRONZE_TABLE}
        GROUP BY 1
        ORDER BY 1
    """)
)

### Step 6 — Silver Layer: Real-Time Enrichment and Fraud Signals

In the medallion architecture, the **Silver layer** transforms raw Bronze data into curated, business-ready datasets. For fintech fraud detection, this means:

| Silver Table | Purpose | Technique |
|---|---|---|
| `txn_events_silver` | Cleaned, deduplicated, enriched transactions | Structured Streaming from Bronze |
| `txn_velocity_1min` | Transaction velocity per sender in 1-min windows | Streaming windowed aggregation with watermarks |
| Channel fingerprinting | Distinct channels per sender — flags multi-channel anomalies | Batch query on Silver enriched table |

**Why streaming?** These Silver tables update every time you run the Auto Loader consumer. In production with a continuous job, they would stay within minutes of the live data — enabling near-real-time fraud alerting.

> **Velocity checks** are a core fraud signal: a legitimate customer rarely makes 3+ transactions to different recipients within 1 minute. High velocity → potential account takeover or card-not-present fraud.

In [0]:
# ---------------------------------------------------------------------------
# Silver layer configuration
# ---------------------------------------------------------------------------
SILVER_SCHEMA     = "silver_layer"
SILVER_TABLE      = f"{CATALOG}.{SILVER_SCHEMA}.txn_events_silver"
VELOCITY_TABLE    = f"{CATALOG}.{SILVER_SCHEMA}.txn_velocity_1min"

# Separate checkpoints for each stream (mandatory — never share checkpoints)
SILVER_CHECKPOINT   = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/_checkpoint_silver"
VELOCITY_CHECKPOINT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/_checkpoint_velocity"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

print(f"\u2705 Silver schema  : {CATALOG}.{SILVER_SCHEMA}")
print(f"\u2705 Silver table   : {SILVER_TABLE}")
print(f"\u2705 Velocity table : {VELOCITY_TABLE}")

In [0]:
# ---------------------------------------------------------------------------
# Silver Enrichment — stream from Bronze, clean, enrich, deduplicate
# ---------------------------------------------------------------------------
from pyspark.sql.functions import (
    col, to_timestamp, hour, dayofweek, when, coalesce, lit, current_timestamp
)

bronze_stream = spark.readStream.table(BRONZE_TABLE)

silver_stream = (
    bronze_stream
    # Parse the ISO-8601 string into a proper TimestampType
    .withColumn("event_ts", to_timestamp("timestamp"))
    # Derived fraud-relevant features
    .withColumn("hour_of_day", hour("event_ts"))
    .withColumn("day_of_week", dayofweek("event_ts"))      # 1=Sun, 7=Sat
    .withColumn("is_high_value", when(col("amount") > 5000, True).otherwise(False))
    .withColumn("is_off_hours",  when(
        (hour("event_ts") < 6) | (hour("event_ts") > 22), True
    ).otherwise(False))
    # Clean nulls
    .withColumn("merchant", coalesce(col("merchant"), lit("N/A")))
    # Drop raw string timestamp (replaced by event_ts)
    .drop("timestamp")
    # Deduplicate on event_id (exactly-once at Silver level)
    .dropDuplicates(["event_id"])
)

query = (
    silver_stream.writeStream
    .format("delta")
    .option("checkpointLocation", SILVER_CHECKPOINT)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(SILVER_TABLE)
)

query.awaitTermination()
print(f"\u2705 Silver enrichment stream completed")
print(f"   Silver table: {SILVER_TABLE}")
print(f"   Row count:    {spark.table(SILVER_TABLE).count():,}")

In [0]:
# ---------------------------------------------------------------------------
# Inspect the Silver enriched table
# ---------------------------------------------------------------------------
print("\U0001F4DD Silver schema (note the derived fraud-signal columns):")
spark.table(SILVER_TABLE).printSchema()

display(
    spark.sql(f"""
        SELECT event_id, event_type, event_ts, amount, currency,
               sender_id, receiver_id, merchant, status, channel,
               hour_of_day, day_of_week, is_high_value, is_off_hours
        FROM {SILVER_TABLE}
        ORDER BY event_ts DESC
        LIMIT 10
    """)
)

#### Velocity Checks — Streaming Windowed Aggregation

We use a **1-minute tumbling window** grouped by `sender_id` to compute transaction velocity. This is a stateful Structured Streaming operation:

* **Watermark** (`1 minute`) — tells Spark how late data can arrive before a window is finalized
* **Tumbling window** (`1 minute`) — non-overlapping time buckets
* **Append output mode** — results are emitted only after the watermark passes the window’s end, guaranteeing the window is complete

Flags generated:
* `is_velocity_alert` — sender made **3+ transactions** in a single 1-min window
* `is_high_volume` — sender moved **> $15,000** in a single 1-min window

In [0]:
# ---------------------------------------------------------------------------
# Velocity Checks — 1-minute windowed aggregation per sender
# ---------------------------------------------------------------------------
from pyspark.sql.functions import (
    col, to_timestamp, window, count, sum as _sum, max as _max,
    avg as _avg, when, current_timestamp
)

# Read Bronze as a stream (separate stream from Silver enrichment)
velocity_source = (
    spark.readStream
    .table(BRONZE_TABLE)
    .withColumn("event_ts", to_timestamp("timestamp"))
    # Watermark: allow up to 1 minute of late data
    .withWatermark("event_ts", "1 minute")
)

# 1-minute tumbling window aggregation per sender
velocity_agg = (
    velocity_source
    .groupBy(
        window("event_ts", "1 minute"),
        "sender_id"
    )
    .agg(
        count("*").alias("txn_count"),
        _sum("amount").alias("total_amount"),
        _max("amount").alias("max_txn_amount"),
        _avg("amount").alias("avg_txn_amount"),
    )
    # Fraud signal flags
    .withColumn("is_velocity_alert",
        when(col("txn_count") >= 2, True).otherwise(False))
    .withColumn("is_high_volume",
        when(col("total_amount") > 15000, True).otherwise(False))
    # Flatten the window struct for easier querying
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "sender_id", "txn_count", "total_amount",
        "max_txn_amount", "avg_txn_amount",
        "is_velocity_alert", "is_high_volume"
    )
)

query = (
    velocity_agg.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", VELOCITY_CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(VELOCITY_TABLE)
)

query.awaitTermination()
print(f"\u2705 Velocity aggregation stream completed")
print(f"   Velocity table: {VELOCITY_TABLE}")
print(f"   Window count:   {spark.table(VELOCITY_TABLE).count():,}")

In [0]:
# ---------------------------------------------------------------------------
# Velocity check results — show highest-velocity senders
# ---------------------------------------------------------------------------
print("\U0001F6A8 Top senders by transaction velocity (1-min windows):\n")

display(
    spark.sql(f"""
        SELECT window_start, window_end, sender_id,
               txn_count, ROUND(total_amount, 2) AS total_amount,
               ROUND(max_txn_amount, 2) AS max_txn,
               is_velocity_alert, is_high_volume
        FROM {VELOCITY_TABLE}
        ORDER BY txn_count DESC, total_amount DESC
        LIMIT 5000
    """)
)

#### Device Fingerprinting — Channel Anomaly Detection

Legitimate users tend to stick to **one or two channels** (e.g., mobile app + web). A sender who suddenly appears across `mobile_app`, `web`, `api`, and `pos_terminal` in a short window is suspicious — it may indicate credential sharing, account takeover, or bot activity.

This query runs as a **batch aggregation** on the Silver enriched table, profiling each sender’s channel diversity over the most recent hour of data.

In [0]:
# ---------------------------------------------------------------------------
# Channel Fingerprinting — detect multi-channel anomalies per sender
# ---------------------------------------------------------------------------
print("\U0001F50D Channel fingerprinting (last 1 hour of Silver data):\n")

display(
    spark.sql(f"""
        WITH sender_channels AS (
            SELECT sender_id,
                   COLLECT_SET(channel) AS channels_used,
                   SIZE(COLLECT_SET(channel)) AS unique_channel_count,
                   COUNT(*) AS txn_count,
                   ROUND(SUM(amount), 2) AS total_amount,
                   COUNT(DISTINCT receiver_id) AS unique_receivers
            FROM {SILVER_TABLE}
            WHERE event_ts >= current_timestamp() - INTERVAL 1 HOUR
            GROUP BY sender_id
        )
        SELECT sender_id,
               channels_used,
               unique_channel_count,
               txn_count,
               total_amount,
               unique_receivers,
               CASE
                   WHEN unique_channel_count >= 3 THEN '⚠️ HIGH RISK'
                   WHEN unique_channel_count  = 2 THEN 'ℹ️ MODERATE'
                   ELSE '\u2705 NORMAL'
               END AS risk_level
        FROM sender_channels
        WHERE txn_count >= 2
        ORDER BY unique_channel_count DESC, txn_count DESC
        LIMIT 200
    """)
)

### Step 7 — AI-Powered Fraud Scoring with `ai_query()`

> **Prerequisite:** Run the [Fraud Detection Model Setup](#notebook-2466812635894406) notebook first to train the model and deploy the `fintech-fraud-scoring` endpoint.

Now we close the loop. The `ai_query()` SQL function calls our **model serving endpoint** to score every Silver transaction in real time. Each row gets a `fraud_score` (0.0–1.0) representing the probability of fraud.

This is the same pattern you'd use in production to:
* Score transactions as they land in Silver
* Feed scores into a Gold alerting table
* Power real-time dashboards and notification systems

```sql
ai_query(
  'fintech-fraud-scoring',                        -- serving endpoint
  named_struct('amount', amount, ...),             -- features as a STRUCT
  returnType => 'DOUBLE'                           -- fraud probability
) AS fraud_score
```

In [0]:
# ---------------------------------------------------------------------------
# Real-time fraud scoring via ai_query() → model serving endpoint
# ---------------------------------------------------------------------------
ENDPOINT_NAME = "fintech-fraud-scoring"

scored_df = spark.sql(f"""
    SELECT event_id,
           event_type,
           event_ts,
           amount,
           sender_id,
           receiver_id,
           channel,
           status,
           hour_of_day,
           day_of_week,
           is_high_value,
           is_off_hours,
           ai_query(
               '{ENDPOINT_NAME}',
               named_struct(
                   'amount',        DOUBLE(amount),
                   'hour_of_day',   DOUBLE(hour_of_day),
                   'day_of_week',   DOUBLE(day_of_week),
                   'is_high_value', DOUBLE(CAST(is_high_value AS INT)),
                   'is_off_hours',  DOUBLE(CAST(is_off_hours AS INT))
               ),
               returnType => 'DOUBLE'
           ) AS fraud_score
    FROM {SILVER_TABLE}
    ORDER BY event_ts DESC
    LIMIT 200
""")

print(f"\U0001F916 Scored {scored_df.count()} transactions via '{ENDPOINT_NAME}'\n")
display(scored_df)

In [0]:
# ---------------------------------------------------------------------------
# Fraud score analysis — risk distribution and high-risk transactions
# ---------------------------------------------------------------------------
print("\U0001F6A8 Fraud score distribution and high-risk transactions:\n")

display(
    spark.sql(f"""
        WITH scored AS (
            SELECT *,
                   ai_query(
                       '{ENDPOINT_NAME}',
                       named_struct(
                           'amount',        DOUBLE(amount),
                           'hour_of_day',   DOUBLE(hour_of_day),
                           'day_of_week',   DOUBLE(day_of_week),
                           'is_high_value', DOUBLE(CAST(is_high_value AS INT)),
                           'is_off_hours',  DOUBLE(CAST(is_off_hours AS INT))
                       ),
                       returnType => 'DOUBLE'
                   ) AS fraud_score
            FROM {SILVER_TABLE}
            LIMIT 500
        )
        SELECT
            CASE
                WHEN fraud_score >= 0.7 THEN '\U0001F534 HIGH (>=0.7)'
                WHEN fraud_score >= 0.3 THEN '\U0001F7E1 MEDIUM (0.3-0.7)'
                ELSE '\U0001F7E2 LOW (<0.3)'
            END AS risk_tier,
            COUNT(*) AS txn_count,
            ROUND(AVG(fraud_score), 4) AS avg_score,
            ROUND(AVG(amount), 2) AS avg_amount,
            ROUND(SUM(amount), 2) AS total_exposure
        FROM scored
        GROUP BY 1
        ORDER BY 1
    """)
)

#### Alternative: UDF-Based Scoring (No Endpoint Required)

The cell below loads the **same fraud model** from Unity Catalog but applies it as a **Spark UDF** instead of calling the serving endpoint. The model runs directly inside the Spark executors — no network hop, no endpoint cost.

| | `ai_query()` + Endpoint | Spark UDF |
|---|---|---|
| **Latency** | HTTP round-trip per batch | In-process, microseconds/row |
| **Throughput** | Throttled by endpoint concurrency | Scales with cluster parallelism |
| **Cost** | Serving endpoint cost | Free — uses existing compute |
| **Multi-consumer** | Any client (APIs, dashboards, SQL) | Only this Spark job |
| **Best for** | Shared service, large models, A/B testing | Batch scoring, small models, cost-sensitive pipelines |

In [0]:
# Only use this if getting mlflow errors if your serverless environment is set to envrionment version 5 it should run fine without it.
# %pip install mlflow

In [0]:
# ---------------------------------------------------------------------------
# UDF-based fraud scoring — same model, no serving endpoint required
# ---------------------------------------------------------------------------
import sys
import mlflow
from pyspark.sql.functions import struct, col

# ---------------------------------------------------------------------------
# Compatibility fix: %pip install mlflow on serverless creates a second
# mlflow module object (pip-installed 3.x) alongside the Databricks wrapper.
# Internal submodules reference the pip version, which is missing several
# attributes the wrapper provides. Replace those references so the internal
# code sees the complete Databricks-patched API.
# ---------------------------------------------------------------------------
# import mlflow.store.artifact.models_artifact_repo as _mar
# _raw = _mar.__dict__.get("mlflow")
# if _raw is not None and _raw is not mlflow:
#     for _mod in sys.modules.values():
#         if _mod is not None and hasattr(_mod, "__dict__"):
#             for _k, _v in list(_mod.__dict__.items()):
#                 if _v is _raw and _k == "mlflow":
#                     _mod.__dict__["mlflow"] = mlflow

MODEL_NAME = f"{CATALOG}.models.fraud_scorer"

# Reference the model by alias — decouples code from specific version numbers.
# Updating the "Champion" alias to a new version automatically rolls it out.
model_uri = f"models:/{MODEL_NAME}@Champion"

# Load from Unity Catalog as a Spark UDF
fraud_udf = mlflow.pyfunc.spark_udf(
    spark,
    model_uri=model_uri,
    result_type="double"
)

# Apply the UDF to the Silver table
silver_df = spark.table(SILVER_TABLE)

scored_udf_df = (
    silver_df
    .withColumn(
        "fraud_score_udf",
        fraud_udf(
            struct(
                col("amount").cast("double").alias("amount"),
                col("hour_of_day").cast("double").alias("hour_of_day"),
                col("day_of_week").cast("double").alias("day_of_week"),
                col("is_high_value").cast("double").alias("is_high_value"),
                col("is_off_hours").cast("double").alias("is_off_hours"),
            )
        ),
    )
    .select(
        "event_id", "event_type", "event_ts", "amount",
        "sender_id", "channel", "is_high_value", "is_off_hours",
        "fraud_score_udf",
    )
    .orderBy(col("fraud_score_udf").desc())
    .limit(200)
)

print(f"\U0001F916 Scored {scored_udf_df.count()} transactions via Spark UDF (no endpoint)\n")
display(scored_udf_df)

### Step 8 — Cleanup

> ⚠️ **Uncomment and run the cell below to remove all demo resources** (Bronze + Silver + Model + Endpoint). Skip if you want to keep exploring.

> **Don’t forget** to also stop the Producer notebook in the other tab!

In [0]:
# ---------------------------------------------------------------------------
# ⚠️ CLEANUP — Uncomment and run to tear down all demo resources
# ---------------------------------------------------------------------------

# # Serving endpoint (not removed by catalog cascade)
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
try:
    w.serving_endpoints.delete(name="fintech-fraud-scoring")
    print("\U0001F5D1\ufe0f  Deleted serving endpoint: fintech-fraud-scoring")
except Exception:
    print("\u2139\ufe0f  Endpoint already deleted or not found")

# Catalog cascade removes all schemas, tables, volumes, and models
spark.sql(f"DROP CATALOG IF EXISTS {CATALOG} CASCADE")
print("\U0001F5D1\ufe0f  Dropped catalog (cascade): fintech_demo")
print("\U0001F9F9 All demo resources removed")

### Key Takeaways

| Concept | What You Learned |
|---|---|
| **Decoupled Producer/Consumer** | The producer (Kinesis sim) and consumer (Auto Loader) run independently — just like in production |
| **Auto Loader (`cloudFiles`)** | Incrementally discovers and ingests new files from cloud storage with exactly-once guarantees |
| **Trigger.AvailableNow** | Processes all pending files in one micro-batch, then stops — ideal for serverless and scheduled jobs |
| **Schema Inference** | No manual DDL needed — Auto Loader reads the JSON structure and infers column types |
| **Schema Evolution** | New upstream fields (`risk_score`) are automatically merged into the Bronze table (`addNewColumns` mode) |
| **Silver Enrichment** | Streaming Bronze → Silver: parse timestamps, derive fraud features (`is_high_value`, `is_off_hours`), deduplicate |
| **Windowed Aggregations** | Tumbling windows + watermarks compute per-sender velocity in near real time |
| **Channel Fingerprinting** | Detecting multi-channel anomalies as a fraud signal using `COLLECT_SET` and distinct channel counts |
| **`ai_query()` Scoring** | Call a model serving endpoint from SQL to score every transaction with a fraud probability in real time |
| **Model Serving** | Deploy an MLflow model from Unity Catalog to a scalable REST endpoint with zero infrastructure management |
| **Checkpointing** | Each streaming table gets its own checkpoint — never share checkpoints between streams |

### Production Next Steps
1. **Replace the simulated Volume path** with an external Volume pointing to your real Kinesis Firehose S3 prefix
2. **Schedule as a continuous Workflow** — `Trigger.AvailableNow` + continuous job trigger for near-real-time ingestion through Silver
3. **Add a Gold alerting table** — write `ai_query()` scored results to a Gold table, trigger alerts on high-risk transactions
4. **A/B test fraud models** — use endpoint traffic splitting to compare model versions with zero downtime
5. **Enable file event notifications** (`cloudFiles.useManagedFileEvents`) for sub-second file discovery at scale
6. **Add data quality expectations** using Lakeflow Declarative Pipelines (formerly DLT) for production guardrails